# Paper 5, RB-6 - one intervention on both encodings: erase the residual stream's positional component, keep the encoding

**Naming.** W1-W7 = September runs. RB = rebuttal runs: RB-0 single heads (`w8_*`, canonical = cuDNN 9.10.2 re-run),
RB-1/1b drift checks, RB-2 axis map, RB-3 intact/shuffled/null, RB-4 mixed RoPE, RB-5 window edge. This is **RB-6**;
it writes to `./rb6_output/`.

**Why (official ICLR reviews).** NpkG W4 and 91UE W1/Q1: the paper's APE-vs-RoPE contrast uses two different
interventions (APE's features removed for good at block 2; RoPE's rotations removed and then restored), so "APE does not
rebuild, RoPE does" may reflect the interventions. NpkG asks for the symmetric test: keep RoPE's rotations, erase or
scramble the positional component of the residual stream at block 4, and see whether it comes back. NpkG W1 names the
counterexample this tests: a RoPE model that keeps position in its residual stream and no longer needs later rotations.
Part A also gives the SAE-free, shape-agnostic positional measure NpkG W5 suggests.

**What is erased.** At block s (the residual stream entering block s, before its LayerNorm), the positional component
is c_s(p) = μ_s(p) − μ̄_s: the mean patch state at grid slot p over 2,000 *fitting* images under RPI (content
shuffled across slots, so the mean depends on the slot only), minus its average over the 196 slots. Fitting images
are disjoint from every evaluation image. Only patch tokens are edited; prefix tokens are untouched.

| arm | edit at block s (patch tokens) | role |
|---|---|---|
| `mean`  | h ← h − c(p) | minimal erasure: equal per-slot means = no linear readout of the slot (Belrose et al., 2023) |
| `scale` | h ← μ̄ + (h − μ(p))·ρ̄/ρ(p) | also equalises each slot's spread (ρ = RMS deviation), so LayerNorm cannot re-expose the slot |
| `sub`   | h ← h − P_S(h − μ̄), S = top-r directions of c (99% of its energy) | removes the positional subspace entirely |
| `randsub` | same as `sub` with a random r-dimensional subspace | control, matched in dimension to `sub` |
| `scramble` | h ← h − c(p) + c(π(p)), π a fixed derangement | moves each slot's component to another slot |
| `rotoff` (RoPE, hybrid) | no edit; rotations off in blocks s-11 | how much position the residual stream keeps without later rotations |
| `<erasure>_rotoff` (RoPE, hybrid) | erasure + rotations off in blocks s-11 | nothing left to rebuild from: must not recover |

s = 4 on every model (the reviewer's block); s = 2 as well on the APE models (the paper's SAE site).

**Reading, fixed before the run** (after a review of the first design):
1. *Erasure check, at the edited state itself* (readout s = the state entering block s after the edit, before block s
   runs): normalised SSDC N_s ≤ 0.10, and on held-out images the exact probe ≤ 4× chance and the row and column probes
   ≤ 1.5× chance. The reading uses the first of `mean`, `scale`, `sub` that passes; if none passes, the site reads
   ERASURE INCOMPLETE.
2. N_b = (SSDC_b − floor_b)/(intact_b − floor_b), only where intact_b − floor_b ≥ 0.10 (otherwise not reported);
   raw SSDC is always reported next to it. Floor: all rotations off (RoPE), embedding zeroed (APE), both (hybrid).
3. R = N_max − N_s, N_max = max N_b over readouts s+1…11 where N_b is defined. REBUILDS if R ≥ 0.20 and N_max ≥ 0.50;
   DOES NOT REBUILD if R < 0.05; otherwise PARTIAL.

Predicted: APE models do not rebuild; RoPE models rebuild. A RoPE model that does not rebuild is NpkG's counterexample.

**Coherence.** Same environment (`env_paper5_cudnn910`), datasets, counts (SSDC/probes: repack 1,000; top-1: ILSVRC
5,000), batch, seed, fp32, readouts and code paths as RB-0 to RB-5. Intact and floor arms must equal RB-0 bit for bit
(RoPE, hybrid). APE models have no RB-0: supervised and DINO are checked against W1b's repack SSDC, and supervised,
AugReg, SAM and DeiT-III against W7's top-1, within stated tolerances (different machine). Runtime ~2.5-3.5 h on one H100.

In [1]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "datasets", "scipy", "pandas", "transformers"]:
    ensure(p)
print("environment ready")

/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


environment ready


In [2]:
import os, json, time, warnings, traceback, shutil, hashlib, platform, socket
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("python", sys.executable)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

python /workspace/NRDL/env_paper5_cudnn910/bin/python
torch 2.10.0+cu128 | timm 1.0.29 | cuda True
device: NVIDIA H100 80GB HBM3


memory: 50.2 / 85.0 GB free


In [3]:
# config
REPO_PATH  = os.environ.get("RB6_REPO", "")
RB0_DIR    = os.environ.get("RB6_RB0_DIR", "")
MODELS_RUN = [m for m in os.environ.get("RB6_MODELS",
              "naver,DINOv3,small,reg1_gap,rope_ape,supervised,dino,augreg,sam,deit3").split(",") if m]
RESUME     = os.environ.get("RB6_RESUME", "1") == "1"
STRICT     = os.environ.get("RB6_STRICT", "1") == "1"
TEST_MODE  = os.environ.get("RB6_TEST", "0") == "1"   # CPU smoke test; needs a test RB-0 made by a seeded W8 copy
OUT_DIR    = Path(os.environ.get("RB6_OUT", "rb6_output"))

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
N_FIT = 2000                                    # fitting images per dataset, taken right after the evaluation images
METRIC, TORCH_SEED, INPUT_SIZE = "manhattan", 20260908, 224
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
ACC_DATASET  = "ILSVRC/imagenet-1k"
N_BLOCKS, FINAL = 12, 12
PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 / RB-0 / RB-2 protocol, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None

SITE_MAIN, SITE_APE_EXTRA = 4, 2
ERASURES = ["mean", "scale", "sub"]             # reading uses the first that passes the erasure check, in this order
EDITS = ERASURES + ["randsub", "scramble"]
SUBSPACE_ENERGY = 0.99
SUBSPACE_SEED, SCRAMBLE_SEED = 20261108, 7
NORM_MIN_RANGE = 0.10                           # N_b only where intact_b - floor_b >= this (raw SSDC always reported)
ERASE_N_MAX, ERASE_EXACT_X, ERASE_AXIS_X = 0.10, 4.0, 1.5
REBUILD_R, REBUILD_NMAX, NOREBUILD_R = 0.20, 0.50, 0.05
DISJOINT_MAX_OVERLAP = 5                        # duplicates allowed between fit and eval sets (ImageNet val has a few exact ones)

EXACT_TOL = 0.0
W1B_TOL_SSDC = 1e-3                             # W1b ran on another machine/session; W1b vs W5 differ by ~2e-5
W7_TOL_POINTS, W7_MIN_AGREE = 0.10, 0.995       # W7 APE ran at batch 128 on the control machine
EXPECT_TORCH, EXPECT_CUDNN = "2.10.0+cu128", 91002
EXPECT_CUDNN_PKGS = {"nvidia-cudnn-cu12": "9.10.2.21"}
FIRST_LAYER_910 = {"naver": 0.00041303808169361526, "DINOv3": 0.0004745740108653513,
                   "small": 0.000510305810057617, "reg1_gap": 0.00026782853342761795}
FIRST_LAYER_RANGE = (1e-4, 2e-3)               # TF32-level kernel for models RB-1b did not record (fp32 would be ~1e-6)

MODEL_SPECS = {
    "naver":      dict(family="rope",   loader="rope", name="vit_base_patch16_rope_224.naver_in1k",        classifier=True,  label="naver ViT-B"),
    "DINOv3":     dict(family="rope",   loader="rope", name="vit_base_patch16_dinov3.lvd1689m",            classifier=False, label="DINOv3 ViT-B"),
    "small":      dict(family="rope",   loader="rope", name="vit_small_patch16_rope_224.naver_in1k",       classifier=True,  label="naver ViT-S"),
    "reg1_gap":   dict(family="rope",   loader="rope", name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k", classifier=True,  label="reg1-gap ViT-B"),
    "rope_ape":   dict(family="hybrid", loader="rope", name="vit_base_patch16_rope_ape_224.naver_in1k",    classifier=True,  label="hybrid ViT-B"),
    "supervised": dict(family="ape",    loader="hf",   name="google/vit-base-patch16-224",                 classifier=True,  label="supervised ViT-B"),
    "dino":       dict(family="ape",    loader="timm", name="vit_base_patch16_224.dino",                   classifier=False, label="DINO ViT-B"),
    "augreg":     dict(family="ape",    loader="timm", name="vit_base_patch16_224.augreg_in1k",            classifier=True,  label="AugReg ViT-B"),
    "sam":        dict(family="ape",    loader="timm", name="vit_base_patch16_224.sam_in1k",               classifier=True,  label="SAM ViT-B"),
    "deit3":      dict(family="ape",    loader="timm", name="deit3_base_patch16_224.fb_in1k",              classifier=True,  label="DeiT-III ViT-B"),
}
def sites_of(key): return [SITE_MAIN, SITE_APE_EXTRA] if MODEL_SPECS[key]["family"] == "ape" else [SITE_MAIN]
def has_rope(key): return MODEL_SPECS[key]["family"] in ("rope", "hybrid")
def floor_name(key): return "both_floor" if MODEL_SPECS[key]["family"] == "hybrid" else "floor"

RB0_SHA256 = {
    "w8_partA_naver.json":     "f50b694d24279ce63b8dac8da1906b7e60ab6522ce8de4c992b1c87ceed10914",
    "w8_partA_DINOv3.json":    "f1d963652ac3867c79bb170da2b2dd86ff8e978008f5ff59304a69947bb909b5",
    "w8_partA_small.json":     "be01ed6297ef819a8fc219853c07fc9a81a76d2c6ffbe9da8501379fdf6acbbb",
    "w8_partA_reg1_gap.json":  "da5ae49a2abc21aa0df8f4c43b249727a7edfe444254e239e637ed96a312fefb",
    "w8_partA_rope_ape.json":  "679d30cbd1b8873d3aea6724f2fa46984dbd5308f167ff0711f51bcdd6150c92",
    "w8_partC_naver.json":     "c983414404ca645bc8144a2bd4ec76bfb583f4c5c9588ec8faffe853e5c3b985",
    "w8_partC_small.json":     "bf895274486dfaa9ab042118e15bb7c9863a5f6201ab27ec11fb539ecf7126f6",
    "w8_partC_reg1_gap.json":  "8b5641235e05b20bfdc597032097b2f85878c228bdc32ec8b63888227e393698",
    "w8_partC_rope_ape.json":  "c0d4b9fec550b3343b52f5f267ceb277b1dcfb81b59efc1d0805734b234bb2bb",
    "w8_perimage_naver.npz":   "5fe4a53ecbf99bc583c56ef6ec9f774c5bd6027e1734dce16fe0c230c36358c6",
    "w8_perimage_small.npz":   "b9f6a3833766b73618c82b3e5ecbb81af672b11eaa49fab503e27bb8734cf40b",
    "w8_perimage_reg1_gap.npz": "10310f0c0109d65f0b594f7774006ddd2c69f5fc74438b13d32956d39738643e",
    "w8_perimage_rope_ape.npz": "73bae58e5d02d1035b597ec1c6e0c613b37a582b8bb2e67b515d9471ddbba192",
}
RB0_FILES = {k: [f"w8_partA_{k}.json"] + ([f"w8_partC_{k}.json", f"w8_perimage_{k}.npz"] if s["classifier"] else [])
             for k, s in MODEL_SPECS.items() if s["family"] != "ape"}
W1B_FILE, W1B_SHA256 = "w1b/w1b_floors_2026-09-09T07-55-22Z.json", "cd776ee5f8e62d9263f5fcb2a583d2aa34a3ecf3aab6ca75703b38df90365669"
W1B_KEY = {"supervised": "APE · google ViT", "dino": "APE · DINOv1"}
W7_APE = {   # (results json, per-image npz) and their sha256
    "supervised": ("w7/w7ape_supervised_2026-09-24T18-12-56Z.json", "18dd342d8b9d2d9f154e87e84062671ddefc833a8dc240b0edf4837dab98555d",
                   "w7/perimage/ape_supervised_correct.npz", "fcd2ee3849f3a47247d346c232d3de61e1596ba4564144c15dcdb075aac66f95"),
    "augreg":     ("w7/w7ape_augreg_2026-09-24T18-12-56Z.json", "06e946e04cc97de6fa79822475e24604e451d2d94301014e867df602a35de50e",
                   "w7/perimage/ape_augreg_correct.npz", "fb877584e1d1c6688b5736ef775ce75563e84655fc0c145bcf41b8f36378b688"),
    "sam":        ("w7/w7ape_sam_MERGED.json", "f8d754fac93bffb10e4652cd712a339637ae658ad4471d49b2603dd0c348dc65",
                   "w7/perimage/ape_sam_correct.npz", "c2e83a69afa8b6005d340a760ee735ad72ca967d35b0a3d046cb3b6f5f498dff"),
    "deit3":      ("w7/w7ape_deit3_2026-09-25T01-02-08Z.json", "0d42773eb22dc2dd4f2bede943eac73d99130f55e6f859a39144fdac557c6f87",
                   "w7/perimage/ape_deit3_correct.npz", "c851a1258715a6476e5d7f9a00e7aa2573266fb29329800ac1edbe1b486ad764"),
}
REPLAY_CHECK_N = 512

RULE_ERASE = (f"Erasure check at the edited state (readout s = entering block s, after the edit): normalised SSDC N_s <= {ERASE_N_MAX}, "
              f"and on held-out images exact probe <= {ERASE_EXACT_X}x chance and row and column probes <= {ERASE_AXIS_X}x chance. "
              f"The reading uses the first of {ERASURES} that passes; none passes = ERASURE INCOMPLETE.")
RULE_NORM = (f"N_b = (SSDC_b - floor_b) / (intact_b - floor_b), only where intact_b - floor_b >= {NORM_MIN_RANGE}; raw SSDC always reported. "
             "Floor = all rotations off (RoPE), embedding zeroed (APE), both (hybrid).")
RULE_READING = (f"R = N_max - N_s, N_max = max N_b over readouts s+1..11 where N_b is defined. REBUILDS if R >= {REBUILD_R} and "
                f"N_max >= {REBUILD_NMAX}; DOES NOT REBUILD if R < {NOREBUILD_R}; otherwise PARTIAL. Predicted: APE does not rebuild, "
                "RoPE rebuilds; a RoPE model that does not rebuild is NpkG's counterexample.")
RULE_CASE = ("Outcome case for the paper (the notebook only labels): case 1 if all four RoPE models rebuild at s=4 and no "
             "APE model does; case 2 if one or two RoPE models do not rebuild (PARTIAL counts as not rebuilding); case 3 (stop and discuss) "
             "if three or more do not; no case if any RoPE model reads ERASURE INCOMPLETE or NOT READABLE; any APE model that rebuilds is flagged.")

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE, N_FIT = 24, 32, 8, 40
    PROBE_PASSES = 2
    OUT_DIR = Path(os.environ.get("RB6_OUT", "rb6_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
_bad = [m for m in MODELS_RUN if m not in MODEL_SPECS]
if _bad: raise SystemExit(f"Unknown entries in RB6_MODELS: {_bad}")

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"rb6_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return bool(passed) or skip
print(f"device {DEVICE} | fp32 | models {MODELS_RUN}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | top-1: {ACC_DATASET} x {N_ACC} | fit: {N_FIT} per dataset | batch {BATCH_SIZE} | seed {TORCH_SEED}")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | strict {STRICT} | test mode {TEST_MODE} | run {RUN_STAMP}")

device cuda | fp32 | models ['naver', 'DINOv3', 'small', 'reg1_gap', 'rope_ape', 'supervised', 'dino', 'augreg', 'sam', 'deit3']
SSDC/probes: benjamin-paine/imagenet-1k-256x256 x 1000 | top-1: ILSVRC/imagenet-1k x 5000 | fit: 2000 per dataset | batch 256 | seed 20260908
output /workspace/NRDL/experiments/rb6/rb6_output | resume True | strict True | test mode False | run 2026-11-08T05-56-19Z


## Pre-flight: are we in `env_paper5_cudnn910`?
Same checks as RB-2 to RB-5. Nothing is measured if any fails.

In [4]:
def _sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as ex:
        return f"(failed: {ex!r})"

def _loaded_cudnn():
    try:
        with open("/proc/self/maps") as f:
            return sorted({l.split()[-1] for l in f if "libcudnn" in l and l.split()[-1].startswith("/")})
    except Exception:
        return []

import importlib.metadata as _md
if DEVICE == "cuda":
    _x = torch.randn(2, 3, 32, 32, device="cuda"); _w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(_x, _w, stride=16); torch.cuda.synchronize(); del _x, _w
CUDNN_PKGS = {}
for _d in _md.distributions():
    _n = (_d.metadata.get("Name") or "").lower()
    if "cudnn" in _n: CUDNN_PKGS[_n] = _d.version
import transformers as _tf
ENV = {
    "python": sys.executable, "prefix": sys.prefix, "host": socket.gethostname(), "platform": platform.platform(),
    "torch": torch.__version__, "timm": timm.__version__, "transformers": _tf.__version__, "numpy": np.__version__,
    "cudnn_reported": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "libcudnn_loaded": _loaded_cudnn(), "cudnn_packages": CUDNN_PKGS,
    "flags": {"matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32, "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
              "cudnn_benchmark": torch.backends.cudnn.benchmark, "cudnn_deterministic": torch.backends.cudnn.deterministic,
              "float32_matmul_precision": torch.get_float32_matmul_precision()},
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "gpu": _sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv") if DEVICE == "cuda" else "cpu",
}
(OUT_DIR / f"rb6_pip_freeze_{RUN_STAMP}.txt").write_text(_sh(f"{sys.executable} -m pip freeze", timeout=300))
(OUT_DIR / f"rb6_environment_{RUN_STAMP}.json").write_text(json.dumps(ENV, indent=1, default=str))
print(json.dumps({k: v for k, v in ENV.items() if k != "gpu"}, indent=1, default=str)); print(ENV["gpu"])

_pre = []
if TEST_MODE:
    gate("pre-flight (cuDNN 9.10.2 environment)", "env", True, "test mode on CPU", skip=True)
else:
    _pre.append(gate("CUDA available", "env", DEVICE == "cuda"))
    _pre.append(gate(f"torch is {EXPECT_TORCH}", "env", torch.__version__ == EXPECT_TORCH, torch.__version__))
    _pre.append(gate(f"PyTorch reports cuDNN {EXPECT_CUDNN} (9.10.2)", "env", ENV["cudnn_reported"] == EXPECT_CUDNN,
                     str(ENV["cudnn_reported"])))
    _prefix = os.path.realpath(sys.prefix)
    _libs = ENV["libcudnn_loaded"]
    _pre.append(gate("every loaded libcudnn file comes from this environment's nvidia/cudnn folder", "env",
                     bool(_libs) and all(os.path.realpath(l).startswith(_prefix) and "/nvidia/cudnn/" in l for l in _libs),
                     "; ".join(Path(l).name for l in _libs) + f" (prefix {_prefix})"))
    _pre.append(gate("the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13)", "env",
                     CUDNN_PKGS == EXPECT_CUDNN_PKGS, str(CUDNN_PKGS)))
    _f = ENV["flags"]
    _pre.append(gate("TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0)", "env",
                     _f["matmul_allow_tf32"] is False and _f["cudnn_allow_tf32"] is True and _f["cudnn_benchmark"] is False
                     and _f["cudnn_deterministic"] is False and _f["float32_matmul_precision"] == "highest", str(_f)))
PREFLIGHT_OK = all(_pre)
if not PREFLIGHT_OK:
    raise SystemExit("PRE-FLIGHT FAILED: this notebook must run in env_paper5_cudnn910 (cuDNN 9.10.2), with PyTorch's "
                     "default flags. Nothing was measured. Switch the kernel / python and run again.")
print("\npre-flight OK")

{
 "python": "/workspace/NRDL/env_paper5_cudnn910/bin/python",
 "prefix": "/workspace/NRDL/env_paper5_cudnn910",
 "host": "gpu-pod",
 "platform": "Linux-6.8.0-41-generic-x86_64-with-glibc2.35",
 "torch": "2.10.0+cu128",
 "timm": "1.0.29",
 "transformers": "5.22.1",
 "numpy": "2.4.6",
 "cudnn_reported": 91002,
 "libcudnn_loaded": [
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_graph.so.9",
  "/workspace/NRDL/env_paper5_cudnn910/lib/python3.11/site-packages/nvidia/cudnn/lib/libcudnn_heuristic.so.9"
 ],
 "cudnn_packages": {
  "nvidia-cudnn-cu12": "9.10.2.21"
 },
 "flags": {
  "matmul_allow_tf32": fal

  [PASS] env: torch is 2.10.0+cu128: 2.10.0+cu128


  [PASS] env: PyTorch reports cuDNN 91002 (9.10.2): 91002


  [PASS] env: every loaded libcudnn file comes from this environment's nvidia/cudnn folder: libcudnn.so.9; libcudnn_engines_precompiled.so.9; libcudnn_engines_runtime_compiled.so.9; libcudnn_graph.so.9; libcudnn_heuristic.so.9 (prefix /workspace/NRDL/env_paper5_cudnn910)


  [PASS] env: the only cuDNN package installed is nvidia-cudnn-cu12 9.10.2.21 (no cu13): {'nvidia-cudnn-cu12': '9.10.2.21'}


  [PASS] env: TF32 / cuDNN flags are PyTorch's defaults (as in W5 and RB-0): {'matmul_allow_tf32': False, 'cudnn_allow_tf32': True, 'cudnn_benchmark': False, 'cudnn_deterministic': False, 'float32_matmul_precision': 'highest'}



pre-flight OK


In [5]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB6_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
REPO_COMMIT = _sh(f"git -C {repo} rev-parse HEAD")
print("repo root:", repo, "| commit", REPO_COMMIT)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope, load_ape, load_ape_timm
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

repo root: /workspace/NRDL/clone/vit-sae-analysis/project_code/src | commit 28e0856cd87c8582aff2783368b3d2f098ae3190


  [PASS] repo-metric: imported metric satisfies its unit test: +1.000/-1.000/-0.005


True

## The canonical RB-0 (fingerprint check) and the APE anchors
RoPE and hybrid arms are gated against RB-0 parts A and C (the **cuDNN 9.10.2 re-run**, `rb0_cudnn910/w8_output`),
checked by sha256. APE models were never in RB-0: their intact and floor SSDC are checked against W1b (supervised, DINO;
repack, same protocol) and their intact top-1 against W7 (supervised, AugReg, SAM, DeiT-III), within the tolerances above.

In [6]:
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

RB0 = {}
_need_rb0 = [k for k in MODELS_RUN if k in RB0_FILES]
if _need_rb0 and not RB0_DIR and not TEST_MODE:
    raise SystemExit("Set RB6_RB0_DIR to the canonical RB-0 folder: "
                     "/workspace/NRDL/experiments/rb0_cudnn910/w8_output")
_rb0 = Path(RB0_DIR).expanduser() if RB0_DIR else None
_rb0_bad = []
for key in _need_rb0:
    for fn in RB0_FILES[key]:
        p = (_rb0 / fn) if _rb0 else None
        if p is None or not p.exists():
            if TEST_MODE: gate(f"RB-0 file {fn} present", key, True, "test mode: no RB-0 folder", skip=True)
            else: gate(f"RB-0 file {fn} present", key, False, str(p)); _rb0_bad.append(fn)
            continue
        if TEST_MODE:
            gate(f"{fn} is the canonical RB-0 file", key, True, "test mode: hash not checked", skip=True)
        else:
            d = sha256_file(p)
            if not gate(f"{fn} is the canonical RB-0 file (cuDNN 9.10.2 re-run)", key, d == RB0_SHA256[fn],
                        f"sha256 {d[:16]}... (expected {RB0_SHA256[fn][:16]}...)"):
                _rb0_bad.append(fn)
if _rb0_bad:
    raise SystemExit(f"RB-0 FINGERPRINT FAILED for {_rb0_bad}. RB6_RB0_DIR must be the cuDNN 9.10.2 re-run "
                     "(rb0_cudnn910/w8_output, 4 Nov). Nothing was measured.")
for key in _need_rb0:
    d = {}
    for part in "AC":
        p = (_rb0 / f"w8_part{part}_{key}.json") if _rb0 else None
        d[part] = json.loads(p.read_text()) if (p is not None and p.exists()) else None
    p = (_rb0 / f"w8_perimage_{key}.npz") if _rb0 else None
    d["P"] = ({n: z_[n] for z_ in [np.load(p)] for n in z_.files} if (p is not None and p.exists()) else None)
    RB0[key] = d
    need = "AC" if MODEL_SPECS[key]["classifier"] else "A"
    gate("RB-0 parts needed by RB-6 are present and complete", key,
         all((d[x] or {}).get("complete") for x in need) and (d["P"] is not None or not MODEL_SPECS[key]["classifier"]),
         ", ".join(f"{x}={'complete' if (d[x] or {}).get('complete') else 'missing'}" for x in need))

RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
W1B, W7A = {}, {}
_ape_run = [k for k in MODELS_RUN if MODEL_SPECS[k]["family"] == "ape"]
if TEST_MODE:
    if _ape_run: gate("committed W1b / W7 APE anchors loaded", "anchors", True, "test mode: tiny random models have no anchors", skip=True)
elif _ape_run:
    f1 = RES_ROOT / W1B_FILE
    if any(k in W1B_KEY for k in _ape_run):
        if gate("committed W1b file found and is the expected one (sha256)", "W1b", f1.exists() and sha256_file(f1) == W1B_SHA256,
                str(f1) + ("" if not f1.exists() else f" sha256 {sha256_file(f1)[:16]}...")):
            e1 = json.load(open(f1))["results"]
            for k, k1 in W1B_KEY.items():
                if k in _ape_run:
                    W1B[k] = {"baseline": e1[k1]["baseline_ssdc"], "floor": e1[k1]["floor_ssdc"]}
    for k in _ape_run:
        if k not in W7_APE: continue
        fj, hj, fi, hi = W7_APE[k]; pj, pi = RES_ROOT / fj, RES_ROOT / fi
        ok = pj.exists() and pi.exists() and sha256_file(pj) == hj and sha256_file(pi) == hi
        if gate("committed W7 APE files (results + per-image) found and are the expected ones (sha256)", k, ok, f"{fj}; {fi}"):
            W7A[k] = {"acc": float(json.load(open(pj))["conditions"]["intact"]["acc"]), "img": np.load(pi)["intact"].astype(np.uint8)}
print("W1b anchors for:", list(W1B), "| W7 top-1 anchors for:", list(W7A))

  [PASS] naver: w8_partA_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f50b694d24279ce6... (expected f50b694d24279ce6...)


  [PASS] naver: w8_partC_naver.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 c983414404ca645b... (expected c983414404ca645b...)


  [PASS] naver: w8_perimage_naver.npz is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 5fe4a53ecbf99bc5... (expected 5fe4a53ecbf99bc5...)


  [PASS] DINOv3: w8_partA_DINOv3.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 f1d963652ac3867c... (expected f1d963652ac3867c...)


  [PASS] small: w8_partA_small.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 be01ed6297ef819a... (expected be01ed6297ef819a...)


  [PASS] small: w8_partC_small.json is the canonical RB-0 file (cuDNN 9.10.2 re-run): sha256 bf895274486dfaa9... (expected bf895274486dfaa9...)


... (14 outputs left out)

  [PASS] augreg: committed W7 APE files (results + per-image) found and are the expected ones (sha256): w7/w7ape_augreg_2026-09-24T18-12-56Z.json; w7/perimage/ape_augreg_correct.npz


  [PASS] sam: committed W7 APE files (results + per-image) found and are the expected ones (sha256): w7/w7ape_sam_MERGED.json; w7/perimage/ape_sam_correct.npz


  [PASS] deit3: committed W7 APE files (results + per-image) found and are the expected ones (sha256): w7/w7ape_deit3_2026-09-25T01-02-08Z.json; w7/perimage/ape_deit3_correct.npz


W1b anchors for: ['supervised', 'dino'] | W7 top-1 anchors for: ['supervised', 'augreg', 'sam', 'deit3']


In [7]:
# datasets: explicit ids, no fallback; fitting images follow the evaluation images
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed, start=0):
            self.n, self.seed, self.start = n, seed, start
        def skip(self, k): return _FakeStream(self.n, self.seed, self.start + k)
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                if i >= self.start: yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
    gate("SSDC dataset is the repack (W2-W6, RB-0 to RB-5)", "data", SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("top-1 dataset is the ILSVRC split (W7, RB-0 to RB-5)", "data", ACC_DATASET == "ILSVRC/imagenet-1k", ACC_DATASET)
FIT_SRC = {"rep": (lambda: DS_SSDC.skip(N_SSDC), f"{SSDC_DATASET} images {N_SSDC}..{N_SSDC + N_FIT - 1}"),
           "acc": (lambda: DS_ACC.skip(N_ACC), f"{ACC_DATASET} images {N_ACC}..{N_ACC + N_FIT - 1}")}

loaded benjamin-paine/imagenet-1k-256x256 [validation]


loaded ILSVRC/imagenet-1k [validation]
  [PASS] data: SSDC dataset is the repack (W2-W6, RB-0 to RB-5): benjamin-paine/imagenet-1k-256x256


  [PASS] data: top-1 dataset is the ILSVRC split (W7, RB-0 to RB-5): ILSVRC/imagenet-1k


In [8]:
# source-aware helpers (timm and transformers)
def pix(im):  return im if torch.is_tensor(im) else im["pixel_values"]
def to_dev(im):
    if torch.is_tensor(im): return im.to(DEVICE)
    from transformers import BatchFeature
    return BatchFeature({k: v.to(DEVICE) for k, v in im.items()})   # BatchFeature.to() moves itself in place
def fwd(model, source, im):
    return model(im) if source == "timm" else model(**im).logits
def clone_im(im):
    if torch.is_tensor(im): return im.clone()
    from transformers import BatchFeature
    return BatchFeature({k: v.clone() for k, v in im.items()})
def _bsz(args, kw):
    if args and torch.is_tensor(args[0]): return int(args[0].shape[0])
    return int(kw["pixel_values"].shape[0])
def _hidden(args, kw):
    if args and torch.is_tensor(args[0]): return args[0]
    if "hidden_states" in kw: return kw["hidden_states"]
    raise RuntimeError("block called without a positional tensor or hidden_states")
def _with_hidden(args, kw, y):
    if args and torch.is_tensor(args[0]): return (y,) + tuple(args[1:]), kw
    kw = dict(kw); kw["hidden_states"] = y; return args, kw
def img_hashes(cache_or_dl, out):
    for im, _ in cache_or_dl:
        t = pix(im)
        for i in range(t.shape[0]): out.append(hashlib.sha1(t[i].contiguous().numpy().tobytes()).hexdigest()[:16])
        yield im, _

In [9]:
# rotary helpers: from W5 / RB-0 / RB-5
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

class RopeGlobal:          # W8 / RB-0's floor: rotations off in every block
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        if s.mod is None: raise RuntimeError("no get_embed module")
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []

class RopeComposite:       # rotations off in a set of blocks (RB-0 / RB-5)
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class APEZero:             # W1b / W8 / RB-0's APE floor: the whole learned embedding set to zero
    def __init__(s, model): s.model = model; s.handles = []
    def __enter__(s):
        cands = [n for n, _ in s.model.named_parameters() if n.endswith("pos_embed") or n.endswith("position_embeddings")]
        if not cands: raise RuntimeError("no positional-embedding parameter")
        s.name = sorted(cands, key=len)[0]; parts = s.name.split("."); m = s.model
        for p in parts[:-1]: m = getattr(m, p)
        s.parent, s.attr = m, parts[-1]
        pe = getattr(s.parent, s.attr); s.saved = pe.detach().clone()
        with torch.no_grad(): pe.zero_()
        s.handles = [1]; return s
    def __exit__(s, *e):
        with torch.no_grad(): getattr(s.parent, s.attr).copy_(s.saved)
        s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

class Multi:               # several interventions at once; hooks are counted across all of them
    def __init__(s, *ctxs): s.ctxs = ctxs; s.handles = []
    def __enter__(s):
        for c in s.ctxs: c.__enter__()
        s.handles = [h for c in s.ctxs for h in (getattr(c, "handles", []) or [])]; return s
    def __exit__(s, *e):
        for c in reversed(s.ctxs): c.__exit__(*e)
        s.handles = []

## The edits
`derive_params` turns the fitted per-slot statistics into the edit parameters; `apply_edit` is the edit itself.
Both are plain functions and are gated on synthetic data below, with the same code the model run uses.

In [10]:
def derangement(P, seed=SCRAMBLE_SEED):
    rng = np.random.default_rng(seed)
    while True:
        p = rng.permutation(P)
        if not np.any(p == np.arange(P)): return p

def derive_params(mu, Vp, site):
    """mu: [P, D] per-slot mean (float64), Vp: [P] per-slot E||h - mu_p||^2. Returns float64 CPU tensors."""
    mu = torch.as_tensor(mu, dtype=torch.float64); Vp = torch.as_tensor(Vp, dtype=torch.float64)
    P, D = mu.shape
    mubar = mu.mean(0); c = mu - mubar
    rho = Vp.clamp_min(1e-12).sqrt(); rhobar = Vp.mean().clamp_min(1e-12).sqrt()
    _, sv, Vh = torch.linalg.svd(c, full_matrices=False)
    e = (sv ** 2).cumsum(0) / (sv ** 2).sum().clamp_min(1e-30)
    r = int(min(int((e < SUBSPACE_ENERGY).sum().item()) + 1, len(sv)))
    V = Vh[:r].T.contiguous()
    g = torch.Generator().manual_seed(SUBSPACE_SEED + site)
    Q, _ = torch.linalg.qr(torch.randn(D, r, generator=g, dtype=torch.float64))
    perm = torch.as_tensor(derangement(P), dtype=torch.long)
    return {"mu": mu, "mubar": mubar, "c": c, "ratio": rhobar / rho, "V": V, "Q": Q.contiguous(), "perm": perm, "r": r,
            "energy_kept_by_V": float(e[r - 1])}

def params_to(p, device, dtype=torch.float32):
    return {k: (v.to(device=device, dtype=dtype) if torch.is_tensor(v) and v.is_floating_point() else
                (v.to(device) if torch.is_tensor(v) else v)) for k, v in p.items()}

def apply_edit(h, kind, p):
    """h: [B, P, D] patch tokens of the residual stream entering the edited block."""
    if kind == "mean":     return h - p["c"]
    if kind == "scale":    return p["mubar"] + (h - p["mu"]) * p["ratio"][:, None]
    if kind == "sub":      return h - ((h - p["mubar"]) @ p["V"]) @ p["V"].T
    if kind == "randsub":  return h - ((h - p["mubar"]) @ p["Q"]) @ p["Q"].T
    if kind == "scramble": return h - p["c"] + p["c"][p["perm"]]
    raise ValueError(kind)

class ResidualEdit:
    """Pre-hook on block s: rewrite the patch tokens entering it (prefix tokens untouched). Readout s sees the edited
    state."""
    def __init__(s, model, source, block, kind, prm, n_prefix):
        s.blk = get_vit_blocks(model, source)[block]; s.kind, s.prm, s.npf = kind, prm, n_prefix; s.handles = []
    def __enter__(s):
        def pre(mod, args, kw):
            x = _hidden(args, kw)
            y = x.clone(); y[:, s.npf:, :] = apply_edit(x[:, s.npf:, :], s.kind, s.prm)
            return _with_hidden(args, kw, y)
        s.handles = [s.blk.register_forward_pre_hook(pre, with_kwargs=True)]; return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

# Part A statistics from the fitting sums (also gated on synthetic data)
def stats_from_sums(S, Se, So, Q2, n, ne, no, H, W):
    """S, Se, So: [P, D] sums (all / even / odd images), Q2: [P] sum of squared norms. float64 numpy."""
    mu = S / n; mubar = mu.mean(0); c = mu - mubar; P = mu.shape[0]
    Vp = Q2 / n - (mu ** 2).sum(1)
    tot = float((Q2 / n).mean() - (mubar ** 2).sum())                      # E ||h - mubar||^2 over tokens
    ec = float((c ** 2).sum(1).mean())
    pvf_raw = ec / tot; pvf_bc = (ec - (1 - 1 / P) * float(Vp.mean()) / n) / tot
    g = c.reshape(H, W, -1); rr = g.mean(1, keepdims=True); cc = g.mean(0, keepdims=True); it = g - rr - cc
    e_r, e_c, e_i = float((rr ** 2).sum() * W), float((cc ** 2).sum() * H), float((it ** 2).sum())
    es = max(e_r + e_c + e_i, 1e-30)
    sv = np.linalg.svd(c, compute_uv=False); cum = np.cumsum(sv ** 2) / max((sv ** 2).sum(), 1e-30)
    r99 = int(min(int((cum < SUBSPACE_ENERGY).sum()) + 1, len(sv)))
    ce = Se / ne - (Se / ne).mean(0); co = So / no - (So / no).mean(0)
    split = float((ce * co).sum() / max(np.sqrt((ce ** 2).sum() * (co ** 2).sum()), 1e-30))
    return {"pvf_raw": pvf_raw, "pvf_bc": pvf_bc, "row_frac": e_r / es, "col_frac": e_c / es, "int_frac": e_i / es,
            "r99": r99, "split_half_cos": split, "total_var": tot, "n": int(n)}, mu, Vp, c

In [11]:
# algebra gates on synthetic data (same functions the model run uses)
_g = torch.Generator().manual_seed(0)
_P, _D, _N = 16, 32, 4000
_ctrue = torch.randn(_P, _D, generator=_g, dtype=torch.float64) * 2.0
_sc = 0.5 + torch.rand(_P, generator=_g, dtype=torch.float64) * 2.0
_X = torch.randn(_N, _P, _D, generator=_g, dtype=torch.float64) * _sc[:, None] + _ctrue + 3.0
_mu = _X.mean(0); _Vp = ((_X - _mu) ** 2).sum(-1).mean(0)
_pr = derive_params(_mu, _Vp, site=4)
_slotmean = lambda Y: Y.mean(0)
_m = apply_edit(_X, "mean", _pr)
gate("mean erasure: every slot's mean equals the average over slots", "edit-algebra",
     torch.allclose(_slotmean(_m), _pr["mubar"].expand(_P, _D), atol=1e-9))
_s = apply_edit(_X, "scale", _pr); _sv = ((_s - _pr["mubar"]) ** 2).sum(-1).mean(0)
gate("mean+scale erasure: equal slot means and equal slot spread", "edit-algebra",
     torch.allclose(_slotmean(_s), _pr["mubar"].expand(_P, _D), atol=1e-9) and torch.allclose(_sv, _sv.mean().expand(_P), rtol=1e-9),
     f"spread range {float(_sv.min()):.4f}-{float(_sv.max()):.4f}")
_u = apply_edit(_X, "sub", _pr); _cu = _slotmean(_u) - _slotmean(_u).mean(0)
gate("subspace erasure removes >= 99% of the positional component's energy", "edit-algebra",
     float((_cu ** 2).sum() / (_pr["c"] ** 2).sum()) <= 1 - SUBSPACE_ENERGY + 1e-9,
     f"r={_pr['r']} of {_D}; energy left {float((_cu ** 2).sum() / (_pr['c'] ** 2).sum()):.2e}")
_Q, _V = _pr["Q"], _pr["V"]
gate("random subspace: orthonormal, same dimension as the positional subspace, reproducible", "edit-algebra",
     _Q.shape == _V.shape and torch.allclose(_Q.T @ _Q, torch.eye(_pr["r"], dtype=torch.float64), atol=1e-10)
     and torch.equal(_Q, derive_params(_mu, _Vp, site=4)["Q"]) and not torch.equal(_Q, derive_params(_mu, _Vp, site=2)["Q"]))
_rr = apply_edit(_X, "randsub", _pr); _cr = _slotmean(_rr) - _slotmean(_rr).mean(0)
gate("random-subspace control leaves most of the positional component (it is a control, not an erasure)", "edit-algebra",
     float((_cr ** 2).sum() / (_pr["c"] ** 2).sum()) > 0.3, f"energy left {float((_cr ** 2).sum() / (_pr['c'] ** 2).sum()):.3f}")
_q = apply_edit(_X, "scramble", _pr); _pm = _pr["perm"]
gate("scramble: slot p carries slot pi(p)'s component; pi has no fixed point", "edit-algebra",
     torch.allclose(_slotmean(_q), _pr["mubar"] + _pr["c"][_pm], atol=1e-9) and not bool((_pm == torch.arange(_P)).any())
     and sorted(_pm.tolist()) == list(range(_P)))
gate("derangement is reproducible", "edit-algebra", np.array_equal(derangement(196), derangement(196)))

class _StubM(nn.Module):
    def __init__(s): super().__init__(); s.blocks = nn.ModuleList([nn.Identity() for _ in range(3)])
_sm = _StubM(); _x = torch.randn(2, 3 + _P, _D, dtype=torch.float64)
with ResidualEdit(_sm, "timm", 1, "mean", _pr, 3) as _re:
    _y = _sm.blocks[1](_x); _y0 = _sm.blocks[0](_x); _nh = len(_re.handles)
gate("edit hook: prefix tokens untouched, patch tokens edited, other blocks untouched, one hook", "edit-algebra",
     torch.equal(_y[:, :3], _x[:, :3]) and torch.allclose(_y[:, 3:], _x[:, 3:] - _pr["c"]) and torch.equal(_y0, _x) and _nh == 1)
gate("edit hook removed on exit", "edit-algebra", torch.equal(_sm.blocks[1](_x), _x))

# Part A statistics: a pure row component, a pure column component, and no component at all
_H = _W = 4; _n = 6000
def _sums(X):
    ev = np.arange(X.shape[0]) % 2 == 0
    return X.sum(0), X[ev].sum(0), X[~ev].sum(0), (X ** 2).sum(-1).sum(0), X.shape[0], int(ev.sum()), int((~ev).sum())
_rng = np.random.default_rng(1)
_row = np.repeat(_rng.normal(size=(_H, 1, 8)), _W, 1).reshape(16, 8) * 1.5
_col = np.repeat(_rng.normal(size=(1, _W, 8)), _H, 0).reshape(16, 8) * 1.5
_A0 = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8))), _H, _W)[0]
_Ar = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8)) + _row), _H, _W)[0]
_Ac = stats_from_sums(*_sums(_rng.normal(size=(_n, 16, 8)) + _col), _H, _W)[0]
_true_r = float(((_row - _row.mean(0)) ** 2).sum(1).mean()); _true_r /= (_true_r + 8.0)
gate("Part A: no positional component gives bias-corrected PVF ~ 0", "partA-algebra", abs(_A0["pvf_bc"]) < 0.002,
     f"raw {_A0['pvf_raw']:.4f}, corrected {_A0['pvf_bc']:+.5f}")
gate("Part A: bias-corrected PVF recovers the true value; row / column split is right", "partA-algebra",
     abs(_Ar["pvf_bc"] - _true_r) < 0.01 and _Ar["row_frac"] > 0.95 and _Ac["col_frac"] > 0.95 and _Ar["split_half_cos"] > 0.9,
     f"PVF {_Ar['pvf_bc']:.4f} vs {_true_r:.4f}; row {_Ar['row_frac']:.3f}, col {_Ac['col_frac']:.3f}")

  [PASS] edit-algebra: mean erasure: every slot's mean equals the average over slots


  [PASS] edit-algebra: mean+scale erasure: equal slot means and equal slot spread: spread range 104.0862-104.0862


  [PASS] edit-algebra: subspace erasure removes >= 99% of the positional component's energy: r=15 of 32; energy left 1.78e-30


  [PASS] edit-algebra: random subspace: orthonormal, same dimension as the positional subspace, reproducible


  [PASS] edit-algebra: random-subspace control leaves most of the positional component (it is a control, not an erasure): energy left 0.529


  [PASS] edit-algebra: scramble: slot p carries slot pi(p)'s component; pi has no fixed point


... (2 outputs left out)

  [PASS] edit-algebra: edit hook removed on exit


  [PASS] partA-algebra: Part A: no positional component gives bias-corrected PVF ~ 0: raw 0.0002, corrected +0.00001


  [PASS] partA-algebra: Part A: bias-corrected PVF recovers the true value; row / column split is right: PVF 0.6165 vs 0.6156; row 1.000, col 1.000


True

In [12]:
# image cache: decode once per model, replay through a real DataLoader (RB-0, source-aware)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(clone_im(im), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_hashes(cache):
    h = []; [None for _ in img_hashes(cache, h)]; return h

In [13]:
# measurement: RB-5's measure(), source-aware, with optional captures for the probes
def final_norm_module(model, source):
    if source == "transformers": return model.vit.layernorm, "vit.layernorm"
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def measure(M, ctx, capture=()):
    """One RPI sweep over the cached repack images under `ctx`: SSDC at every block input and at FINAL, plus captured
    patch tokens."""
    model, source, n_prefix = M["model"], M["source"], M["n_prefix"]
    cache = M["cache"]["ssdc"]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    cap = {r: [] for r in capture}
    def keep(r, tok):
        if r in cap: cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())
    extra = [model.register_forward_pre_hook(
        lambda m, args, kw: st.update(cur=st["next"], next=st["next"] + _bsz(args, kw)), with_kwargs=True)]
    for i in range(N_BLOCKS):
        if i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: keep(i, inp[0].detach())))
    def final_hook(m, inp, out):
        o = out[0] if isinstance(out, tuple) else out
        t = fnorm(o.detach()); acc.add(FINAL, t); keep(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    t0 = time.time()
    torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"]),
           "runtime_s": round(time.time() - t0, 1)}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    return res

# probes: W5's per-block-seeded protocol from RB-0 (no noise extras)
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        peak = max(peak, correct / Xte.shape[0])
    del head, opt
    return float(peak)

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block):
    """Exact / row / column probes, trained on 80% of the evaluation images and scored on the other 20%."""
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, _ = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name, (lab, nc) in labels.items():
        y = lab.repeat(n_images)
        out[name] = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

In [14]:
# top-1 on the cached ILSVRC images: RB-0's accuracy(), source-aware
def accuracy(M, ctx):
    model, source, cache = M["model"], M["source"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = fwd(model, source, to_dev(images)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

# the fit: per-slot sums of the stream entering every block, under RPI
def fit_pass(M, tag):
    """Streams N_FIT fitting images (never cached) through predict(RPI=True) with a pre-hook on every block."""
    model, source, npf = M["model"], M["source"], M["n_prefix"]
    blocks = get_vit_blocks(model, source); A = {}; st = {"cur": 0, "next": 0}; hashes = []
    def mpre(m, args, kw):
        st.update(cur=st["next"], next=st["next"] + _bsz(args, kw))
    def bpre(b):
        def f(mod, args, kw):
            x = _hidden(args, kw)[:, npf:, :].detach().double()
            idx = torch.arange(st["cur"], st["cur"] + x.shape[0], device=x.device); ev = (idx % 2) == 0
            if b not in A:
                z = lambda *s: torch.zeros(*s, dtype=torch.float64, device=x.device)
                A[b] = {"S": z(*x.shape[1:]), "Se": z(*x.shape[1:]), "So": z(*x.shape[1:]), "Q2": z(x.shape[1])}
            a = A[b]; a["S"] += x.sum(0); a["Se"] += x[ev].sum(0); a["So"] += x[~ev].sum(0); a["Q2"] += (x * x).sum(-1).sum(0)
        return f
    hs = [model.register_forward_pre_hook(mpre, with_kwargs=True)]
    hs += [blk.register_forward_pre_hook(bpre(b), with_kwargs=True) for b, blk in enumerate(blocks)]
    dl = prep_data(FIT_SRC[tag][0](), M["processor"], source, corruption_type=None, number_images=N_FIT,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    t0 = time.time()
    torch.manual_seed(TORCH_SEED + 1); np.random.seed((TORCH_SEED + 1) % (2 ** 31))
    try:
        predict(model, img_hashes(dl, hashes), source, RPI=True, half=False)
    finally:
        for h in hs: h.remove()
    n = st["next"]; ne = (n + 1) // 2; no = n // 2
    out = {b: {k: v.cpu().numpy() for k, v in a.items()} for b, a in A.items()}
    print(f"    fit [{tag}] {n} images in {time.time() - t0:.0f}s")
    return out, n, ne, no, hashes

In [15]:
# per-model setup
def _smooth_pe(pe, G=14, scale=3.0):
    """Test mode only: a smooth 2-D sin-cos table on the patch slots, so a tiny random APE model has spatial structure."""
    D = pe.shape[-1]; q = D // 4; om = 1.0 / (100.0 ** (torch.arange(q, dtype=torch.float64) / q))
    ii, jj = torch.meshgrid(torch.arange(G, dtype=torch.float64), torch.arange(G, dtype=torch.float64), indexing="ij")
    a, b = ii.reshape(-1, 1) * om, jj.reshape(-1, 1) * om
    t = torch.cat([a.sin(), a.cos(), b.sin(), b.cos()], 1)
    pe.zero_(); pe[0, -G * G:, :t.shape[1]] = (t * scale).to(pe.dtype)

TEST_INIT_SEED = {"naver": 101, "DINOv3": 102, "small": 103, "reg1_gap": 104}   # rope_ape: 0, as in the test W8
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        ncls = 10 if spec["classifier"] else 0
        if spec["loader"] == "hf":
            from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor
            torch.manual_seed(201)
            cfg = ViTConfig(hidden_size=96, num_hidden_layers=12, num_attention_heads=6, intermediate_size=192,
                            image_size=INPUT_SIZE, patch_size=16, num_labels=10)
            model = ViTForImageClassification(cfg).to(DEVICE).eval()
            with torch.no_grad(): _smooth_pe(model.vit.embeddings.position_embeddings)
            return model, ViTImageProcessor(size={"height": INPUT_SIZE, "width": INPUT_SIZE}), "transformers"
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True, num_classes=ncls)
        if key == "naver": tiny["embed_dim"] = 192
        torch.manual_seed(TEST_INIT_SEED.get(key, 0) if spec["family"] != "ape" else 200 + list(MODEL_SPECS).index(key))
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                if spec["family"] == "ape":
                    for ln in ("ls1", "ls2"):
                        g_ = getattr(getattr(blk, ln, None), "gamma", None)
                        if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
            if spec["family"] == "ape": _smooth_pe(model.pos_embed)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    if spec["loader"] == "rope": return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)
    if spec["loader"] == "hf":   return load_ape(spec["name"], device=DEVICE, half=False)
    return load_ape_timm(spec["name"], device=DEVICE, half=False)

def first_layer_rel_diff(M):
    model, source = M["model"], M["source"]
    conv = get_patch_embed_conv(model, source); grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    try:
        with torch.no_grad(): fwd(model, source, to_dev(M["cache"]["ssdc"][0][0]))
    finally:
        h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        v = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
    n = int(grab["x"].shape[0]); del grab, y64
    return v, n

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    n_prefix = 1 if source == "transformers" else int(getattr(model, "num_prefix_tokens", 1))
    if has_rope(key):
        rh, _, nh = attn_attrs(model, source)
    else:
        nh = model.config.num_attention_heads if source == "transformers" else model.blocks[0].attn.num_heads
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "n_heads": int(nh), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    if spec["classifier"]:
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    M["eval_hashes"] = {"rep": cache_hashes(M["cache"]["ssdc"])}
    if spec["classifier"]: M["eval_hashes"]["acc"] = cache_hashes(M["cache"]["acc"])
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(pix(M["cache"]["ssdc"][0][0])[:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg)
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model, source); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    print(f"    source {source} | prefix tokens {n_prefix} | heads {nh} | final norm {fn_name}")
    ra = (RB0.get(key) or {}).get("A")
    if ra:
        gate("num_heads matches RB-0", key, M["n_heads"] == ra["n_heads"], f"{M['n_heads']} vs {ra['n_heads']}")
    if has_rope(key):
        _, rmod = find_rope(model)
        emb0 = rmod.get_embed(shape=(Hg, Wg))
        gate("rope rows equal patch tokens (196) and the table has no head dimension", key,
             int(emb0.shape[0]) == 196 and emb0.dim() == 2, f"shape {tuple(emb0.shape)}")
    else:
        try:
            with APEZero(model) as _z: nm = _z.name
            gate("APE floor finds the learned position embedding", key, True, nm)
        except Exception as ex:
            gate("APE floor finds the learned position embedding", key, False, repr(ex))
    if DEVICE == "cuda" and not TEST_MODE:
        v, nb = first_layer_rel_diff(M); M["first_layer_rel_diff"] = v
        if key in FIRST_LAYER_910:
            exp_ = FIRST_LAYER_910[key]
            gate("first-layer numerics equal RB-1b's cuDNN 9.10.2 value (TF32-level kernel, as in September)", key,
                 nb == BATCH_SIZE and abs(v / exp_ - 1) <= 0.01, f"{v:.4e} vs {exp_:.4e} (bit-identical: {v == exp_}; batch {nb})")
        else:
            gate(f"first-layer numerics are TF32-level, as for every model in RB-1b ({FIRST_LAYER_RANGE[0]:.0e}-{FIRST_LAYER_RANGE[1]:.0e})", key,
                 nb == BATCH_SIZE and FIRST_LAYER_RANGE[0] <= v <= FIRST_LAYER_RANGE[1], f"{v:.4e} (batch {nb})")
    else:
        gate("first-layer numerics", key, True, "test mode / CPU", skip=True)
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    for nm_, mk in [("baseline", NullCtx), ("floor", lambda: floor_ctx(M, floor_name(key)))]:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EXACT_TOL, f"max |delta| {d:.2e}")
    if "acc" in M["cache"]:
        n_chk = min(REPLAY_CHECK_N, N_ACC)
        dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                       batch_size=BATCH_SIZE, half=False, num_workers=0)
        cs = []
        with torch.inference_mode():
            for im, lb in dl:
                cs.append(fwd(model, source, to_dev(im)).argmax(-1).cpu() == torch.as_tensor(lb))
        cs = torch.cat(cs).numpy().astype(np.uint8)
        _, cc, _ = accuracy(M, NullCtx())
        gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key,
             np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    return M

def floor_ctx(M, which):
    model, source = M["model"], M["source"]
    if which == "floor":      return RopeGlobal(model, IDEN) if has_rope(M["key"]) else APEZero(model)
    if which == "ape_floor":  return APEZero(model)
    if which == "both_floor": return Multi(RopeGlobal(model, IDEN), APEZero(model))
    raise ValueError(which)

In [16]:
# bookkeeping and the fixed plan of arms
RESULTS = {}
CFG_FP = {"n_ssdc": N_SSDC, "n_acc": N_ACC, "n_fit": N_FIT, "batch": BATCH_SIZE, "seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET,
          "acc_dataset": ACC_DATASET, "metric": METRIC, "edits": EDITS, "subspace_energy": SUBSPACE_ENERGY,
          "subspace_seed": SUBSPACE_SEED, "scramble_seed": SCRAMBLE_SEED, "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC],
          "test_mode": TEST_MODE}
SESSION = {"session": RUN_STAMP, "torch": torch.__version__, "cudnn": ENV["cudnn_reported"], "python": sys.executable,
           "gpu": ENV["gpu"], "repo_commit": REPO_COMMIT}
def model_file(key): return OUT_DIR / f"rb6_{key}.json"
def img_file(key):   return OUT_DIR / f"rb6_perimage_{key}.npz"
def fit_file(key):   return OUT_DIR / f"rb6_fit_{key}.npz"

def load_R(key):
    f = model_file(key)
    if not (RESUME and f.exists()): return None, {}
    R = json.loads(f.read_text())
    if R.get("config") != CFG_FP:
        raise SystemExit(f"{f.name} was written with a different configuration; move it away or set RB6_RESUME=0.")
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    for n in R["acc"]:
        if n not in P: raise SystemExit(f"{img_file(key).name} lacks per-image results for {n}; set RB6_RESUME=0.")
    return R, P

def save_R(key, R, P):
    R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    if P:
        tmpi = img_file(key).with_name(img_file(key).stem + ".tmp.npz"); np.savez_compressed(tmpi, **P); os.replace(tmpi, img_file(key))
    tmp = model_file(key).with_suffix(".json.tmp"); tmp.write_text(json.dumps(R, indent=1, default=float)); os.replace(tmp, model_file(key))

def floors_of(key):
    return ["floor", "ape_floor", "both_floor"] if MODEL_SPECS[key]["family"] == "hybrid" else ["floor"]

def plan(key):
    """name -> (kind, spec, probe readouts). Same names in R['ssdc'] and R['acc']."""
    sites = sites_of(key); arms = {}
    union = sorted({r for s in sites for r in (s, s + 1, 11)})
    def add(name, spec, ro):
        arms[("ssdc", name)] = (spec, ro)
        if MODEL_SPECS[key]["classifier"]: arms[("acc", name)] = (spec, ())
    add("baseline", ("null",), union)
    for f in floors_of(key): add(f, ("floor", f), union if f == floor_name(key) else ())
    for s in sites:
        for e in EDITS:
            ro = (s, s + 1, 11) if e in ERASURES else ((s, 11) if e == "randsub" else (s,))
            add(f"{e}_s{s}", ("edit", e, s), ro)
        if has_rope(key):
            add(f"rotoff_s{s}", ("rotoff", s), (11,))
            for e in ERASURES: add(f"{e}_rotoff_s{s}", ("edit_rotoff", e, s), ())
    return arms

def expected_hooks(spec):
    k = spec[0]
    if k == "null": return 0
    if k == "floor": return 2 if spec[1] == "both_floor" else 1
    if k == "edit": return 1
    if k == "rotoff": return N_BLOCKS - spec[1]
    if k == "edit_rotoff": return N_BLOCKS - spec[2] + 1
    raise ValueError(spec)

def build_ctx(M, spec, kind):
    model, source = M["model"], M["source"]
    if spec[0] == "null": return NullCtx()
    if spec[0] == "floor": return floor_ctx(M, spec[1])
    tag = "rep" if kind == "ssdc" else "acc"
    if spec[0] == "edit":
        _, e, s = spec; return ResidualEdit(model, source, s, e, M["prm"][tag][s], M["n_prefix"])
    if spec[0] == "rotoff":
        return RopeComposite(model, source, [(list(range(spec[1], N_BLOCKS)), IDEN)])
    if spec[0] == "edit_rotoff":
        _, e, s = spec
        return Multi(ResidualEdit(model, source, s, e, M["prm"][tag][s], M["n_prefix"]),
                     RopeComposite(model, source, [(list(range(s, N_BLOCKS)), IDEN)]))
    raise ValueError(spec)

## The fit and Part A
One pass per dataset over 2,000 fitting images under RPI, never cached and never used for evaluation (checked by
hashing every preprocessed image). The per-slot means and spreads at the edit sites are saved to `rb6_fit_<model>.npz`
and every edit is built from that saved file, so a resumed run uses exactly the same edit. Part A (all 12 blocks):
positional variance fraction (raw and corrected for finite-sample noise), its row / column / interaction split, how many
directions hold 99% of it, split-half reliability, and agreement between the repack and the ILSVRC fits.

In [17]:
def run_fit(M, R):
    key = M["key"]; tags = ["rep"] + (["acc"] if MODEL_SPECS[key]["classifier"] else []); H, W = M["grid"]
    if "partA" in R and fit_file(key).exists():
        d = sha256_file(fit_file(key))
        if not gate("saved fit file is the one this result file was built from (sha256)", key, d == R["fit_sha256"], d[:16] + "..."):
            raise SystemExit(f"{fit_file(key).name} does not match {model_file(key).name}; set RB6_RESUME=0.")
        for t in tags:
            ov = len(set(M["eval_hashes"][t]) & set(R["fit_hashes"][t]))
            gate(f"fitting images [{t}] are disjoint from the evaluation images (saved hashes)", key, ov <= DISJOINT_MAX_OVERLAP, f"{ov} shared")
    else:
        part, cs, save, hashes = {}, {}, {}, {}
        for t in tags:
            sums, n, ne, no, hs = fit_pass(M, t); hashes[t] = hs
            gate(f"fit [{t}] saw {N_FIT} images ({FIT_SRC[t][1]})", key, n == N_FIT and len(hs) == N_FIT, f"{n}")
            ov = len(set(M["eval_hashes"][t]) & set(hs))
            gate(f"fitting images [{t}] are disjoint from the evaluation images", key, ov <= DISJOINT_MAX_OVERLAP,
                 f"{ov} of {N_FIT} identical to an evaluation image (allowed {DISJOINT_MAX_OVERLAP})")
            part[t] = {}; cs[t] = {}
            for b in range(N_BLOCKS):
                a = sums[b]
                st_, mu, Vp, c = stats_from_sums(a["S"], a["Se"], a["So"], a["Q2"], n, ne, no, H, W)
                part[t][str(b)] = st_; cs[t][b] = c
                if b in sites_of(key):
                    save[f"{t}_mu_b{b}"] = mu; save[f"{t}_Vp_b{b}"] = Vp
        if "acc" in cs:
            part["cross_cos"] = {str(b): float((cs["rep"][b] * cs["acc"][b]).sum() /
                                               max(np.sqrt((cs["rep"][b] ** 2).sum() * (cs["acc"][b] ** 2).sum()), 1e-30))
                                 for b in range(N_BLOCKS)}
        np.savez(fit_file(key), **save)
        R["partA"] = part; R["fit_sha256"] = sha256_file(fit_file(key)); R["fit_hashes"] = hashes
        R["fit_sources"] = {t: FIT_SRC[t][1] for t in tags}
    z = np.load(fit_file(key)); M["prm"] = {}; R["edit_params"] = {}
    for t in tags:
        M["prm"][t] = {}
        for s in sites_of(key):
            p = derive_params(z[f"{t}_mu_b{s}"], z[f"{t}_Vp_b{s}"], site=s)
            M["prm"][t][s] = params_to(p, DEVICE)
            R["edit_params"][f"{t}_s{s}"] = {"subspace_dim": p["r"], "energy_in_subspace": p["energy_kept_by_V"],
                                             "spread_ratio_range": [float(p["ratio"].min()), float(p["ratio"].max())],
                                             "positional_norm_mean": float(p["c"].norm(dim=1).mean())}
    print("    edit parameters:", json.dumps(R["edit_params"]))

## Gates on the saved results
All read the saved file, so a resumed or re-loaded run still checks every arm. "Bit for bit" = identical values.

In [18]:
def _maxdiff(a, b): return max((abs(x - y) for x, y in zip(a, b)), default=0.0)
C_NAME = {"baseline": "baseline", "floor": "floor", "ape_floor": "ape_off", "both_floor": "both_off"}

def check_refs(key, R, P):
    ok = []; s = R["ssdc"]; fam = MODEL_SPECS[key]["family"]
    if fam != "ape":
        ca = RB0[key]["A"]["conditions"]
        for n in ["baseline"] + floors_of(key):
            d = _maxdiff(s[n]["ssdc"], ca[n]["ssdc"])
            ok.append(gate(f"{n} SSDC equals RB-0 part A bit for bit (13 readouts)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}"))
        if MODEL_SPECS[key]["classifier"]:
            cc, Pc = RB0[key]["C"]["conditions"], RB0[key]["P"]
            for n in ["baseline"] + floors_of(key):
                good = R["acc"][n]["acc"] == cc[C_NAME[n]]["acc"] and np.array_equal(P[n], Pc[C_NAME[n]])
                ok.append(gate(f"{n} top-1 equals RB-0 part C '{C_NAME[n]}' bit for bit (mean and per image)", key, good,
                               f"{R['acc'][n]['acc'] * 100:.2f} vs {cc[C_NAME[n]]['acc'] * 100:.2f}"))
    else:
        if key in W1B:
            for n, k1 in (("baseline", "baseline"), ("floor", "floor")):
                d = _maxdiff(s[n]["ssdc"][:N_BLOCKS], W1B[key][k1])
                ok.append(gate(f"{n} SSDC agrees with W1b (repack, 1,000 images; W1b ran fp16 on another day) within {W1B_TOL_SSDC}", key,
                               d <= W1B_TOL_SSDC, f"max |delta| {d:.1e}; block 11 {s[n]['ssdc'][11]:.4f} vs {W1B[key][k1][11]:.4f}"))
        elif not TEST_MODE:
            gate("SSDC anchor: none for this model (W1b measured supervised and DINO only); same code path as the anchored models", key, True)
        if key in W7A:
            w = W7A[key]; n7 = len(w["img"]); ours = P["baseline"][:n7]
            a7, ao = w["acc"] * 100, float(ours.mean() * 100); agree = float((ours == w["img"]).mean())
            ok.append(gate(f"intact top-1 agrees with W7 on the same first {n7} ILSVRC images (W7: batch 128, other session)", key,
                           abs(ao - a7) <= W7_TOL_POINTS and agree >= W7_MIN_AGREE,
                           f"{ao:.2f} vs {a7:.2f}; per-image agreement {agree * 100:.2f}%"))
    sb, sf = s["baseline"]["ssdc"], s[floor_name(key)]["ssdc"]
    _d = ", ".join(f"b{x}: {sb[x]:.3f} -> {sf[x]:.3f}" for x in sites_of(key))
    if TEST_MODE and fam != "ape":
        gate("floor lowers SSDC at every edit site", key, True, "test mode: tiny random RoPE models have no spatial structure; " + _d, skip=True)
    else:
        ok.append(gate("floor lowers SSDC at every edit site", key, all(sf[x] < sb[x] for x in sites_of(key)), _d))
    return all(ok)

def check_arms(key, R, P):
    arms = plan(key); s = R["ssdc"]
    miss = [f"{k}:{n}" for (k, n) in arms if n not in R[k]]
    gate("every planned arm measured", key, not miss, f"{len(arms) - len(miss)}/{len(arms)}" + (f"; missing {miss[:4]}" if miss else ""))
    badh = [f"{k}:{n}" for (k, n), (spec, _) in arms.items() if n in R[k] and R[k][n]["hooks"] != expected_hooks(spec)]
    gate("every arm registered exactly the expected hooks", key, not badh, ",".join(badh[:5]))
    badn = [f"{k}:{n}" for (k, n) in arms if n in R[k] and R[k][n]["n_images"] != (N_ACC if k == "acc" else N_SSDC)]
    gate("every arm saw every image", key, not badn, ",".join(badn[:5]))
    badp = [n for (k, n), (_, ro) in arms.items() if k == "ssdc" and n in s and sorted(map(int, s[n].get("probes", {}))) != sorted(ro)]
    gate("every arm has its planned probes", key, not badp, ",".join(badp[:5]))
    base = s["baseline"]["ssdc"]; w_pre, w_inert, w_rot = 0.0, [], 0.0
    for (k, n), (spec, _) in arms.items():
        if k != "ssdc" or n not in s: continue
        v = s[n]["ssdc"]
        if spec[0] in ("edit", "edit_rotoff"):
            st_ = spec[2]; w_pre = max(w_pre, _maxdiff(v[:st_], base[:st_]))
            if v[st_] == base[st_]: w_inert.append(n)
        if spec[0] == "rotoff":
            st_ = spec[1]; w_rot = max(w_rot, _maxdiff(v[:st_ + 1], base[:st_ + 1]))
            if v[FINAL] == base[FINAL]: w_inert.append(n)
    gate("edits leave every readout before the edited block bit-identical to intact", key, w_pre <= EXACT_TOL, f"max |delta| {w_pre:.1e}")
    gate("every edit changes the edited state (readout s), every rotations-off arm changes the output", key, not w_inert, ",".join(w_inert[:5]))
    if has_rope(key):
        gate("rotations off in blocks s-11 leave readouts 0..s bit-identical to intact (rotations act inside attention)", key,
             w_rot <= EXACT_TOL, f"max |delta| {w_rot:.1e}")
        d = max(abs(s[f"{e}_rotoff_s{x}"]["ssdc"][x] - s[f"{e}_s{x}"]["ssdc"][x]) for x in sites_of(key) for e in ERASURES)
        gate("erasure + rotations off has the same edited state as the erasure alone (readout s)", key, d <= EXACT_TOL, f"max |delta| {d:.1e}")
    if not TEST_MODE:
        ses = R.get("sessions", [])
        gate("every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002", key,
             bool(ses) and all(x.get("cudnn") == EXPECT_CUDNN and x.get("torch") == EXPECT_TORCH for x in ses), f"{len(ses)} session(s)")

## Measurement
Per model: fit and Part A, then intact and floor arms and their gates (a failure stops that model when
`RB6_STRICT=1`), then every planned arm. Saves after every arm; a re-run continues where it stopped.

In [19]:
def run_model(M):
    key = M["key"]
    R, P = load_R(key)
    if R is None:
        R, P = {"model": key, "name": MODEL_SPECS[key]["name"], "family": MODEL_SPECS[key]["family"], "notebook": "RB-6",
                "config": CFG_FP, "n_heads": M["n_heads"], "n_prefix": M["n_prefix"], "sites": sites_of(key),
                "acc": {}, "ssdc": {}, "sessions": []}, {}
    else:
        print(f"  resuming {model_file(key).name}: {len(R['acc'])} top-1 arms, {len(R['ssdc'])} SSDC arms saved")
    R["sessions"].append({**SESSION, "first_layer_rel_diff": M.get("first_layer_rel_diff")})
    run_fit(M, R); save_R(key, R, P)
    fit_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if fit_fail and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: a fit gate failed (RB6_STRICT=1). Send the gate report."); return R
    arms = plan(key); t0 = time.time(); n_new = [0]; H, W = M["grid"]
    def run(k, name):
        spec, ro = arms[(k, name)]
        if name in R[k]: return
        ctx = build_ctx(M, spec, k)
        if k == "acc":
            a, c, hooks = accuracy(M, ctx); R["acc"][name] = {"acc": a, "hooks": hooks, "n_images": int(len(c)), "spec": list(spec)}; P[name] = c
        else:
            r = measure(M, ctx, capture=tuple(ro))
            pr = {str(b): probe_block(r["acts"][b], H, W, b) for b in ro}
            R["ssdc"][name] = {"ssdc": r["ssdc"], "hooks": r["hooks"], "n_images": r["n_images"], "spec": list(spec), "probes": pr}
            del r
        n_new[0] += 1; save_R(key, R, P)
    refs = [(k, n) for (k, n) in arms if n in ["baseline"] + floors_of(key)]
    for k, n in refs: run(k, n)
    if not check_refs(key, R, P) and STRICT and not TEST_MODE:
        gate("arms measured", key, False, "STOPPED: an intact / floor gate failed (RB6_STRICT=1). Send the gate report.")
        return R
    todo = [(k, n) for (k, n) in arms if (k, n) not in refs and n not in R[k]]
    print(f"\n--- {key}: {len(todo)} arms to measure ---", flush=True)
    for i, (k, n) in enumerate(todo):
        run(k, n)
        el = time.time() - t0
        print(f"    {i + 1}/{len(todo)} {k}:{n} | {el / 60:.1f} min, ~{el / (i + 1) * (len(todo) - i - 1) / 60:.0f} min left", flush=True)
    R["complete"] = all(n in R[k] for (k, n) in arms)
    save_R(key, R, P)
    check_arms(key, R, P)
    gate("model finished", key, True, f"{n_new[0]} new arms, {(time.time() - t0) / 60:.1f} min; complete {R['complete']}")
    return R

In [20]:
T_RUN = time.time()
for key in MODELS_RUN:
    if MODEL_SPECS[key]["family"] != "ape" and key not in RB0:
        gate("model measured", key, False, "no RB-0 files for this model"); continue
    R0, P0 = load_R(key)
    if R0 and R0.get("complete"):
        print(f"\n{key}: already complete in {model_file(key).name}; loading and re-checking")
        RESULTS[key] = R0; check_refs(key, R0, P0); check_arms(key, R0, P0); continue
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    gate("model setup completed", key, True)
    setup_fail = [g for g in GATES if g["scope"] == key and g["status"] == "FAIL"]
    if setup_fail and STRICT and not TEST_MODE:
        gate("model measured", key, False, f"STOPPED: {len(setup_fail)} setup gate(s) failed: " + "; ".join(g["gate"] for g in setup_fail)[:300])
        del M; continue
    try:
        RESULTS[key] = run_model(M)
    except SystemExit:
        raise
    except Exception as ex:
        traceback.print_exc(); gate("model measured", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nmeasurement finished in {(time.time() - T_RUN) / 60:.1f} min")


naver: already complete in rb6_naver.json; loading and re-checking
  [PASS] naver: baseline SSDC equals RB-0 part A bit for bit (13 readouts): max |delta| 0.0e+00


  [PASS] naver: floor SSDC equals RB-0 part A bit for bit (13 readouts): max |delta| 0.0e+00


  [PASS] naver: baseline top-1 equals RB-0 part C 'baseline' bit for bit (mean and per image): 84.64 vs 84.64


  [PASS] naver: floor top-1 equals RB-0 part C 'floor' bit for bit (mean and per image): 38.84 vs 38.84


  [PASS] naver: floor lowers SSDC at every edit site: b4: 0.469 -> 0.013


  [PASS] naver: every planned arm measured: 22/22


... (153 outputs left out)

  [PASS] deit3: edits leave every readout before the edited block bit-identical to intact: max |delta| 0.0e+00


  [PASS] deit3: every edit changes the edited state (readout s), every rotations-off arm changes the output


  [PASS] deit3: every session that wrote this file ran torch 2.10.0+cu128 with cuDNN 91002: 1 session(s)



measurement finished in 10.3 min


## Analysis
Reads only saved files. Every number below is computed by the fixed rules printed at the top.

In [21]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean() * 100), float(d.std(ddof=1) / np.sqrt(len(d)) * 100)
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))
def r4(x): return None if x is None else round(float(x), 4)

def norm_curve(v, base, flo):
    return [((v[b] - flo[b]) / (base[b] - flo[b])) if (base[b] - flo[b]) >= NORM_MIN_RANGE else None for b in range(N_BLOCKS + 1)]

def erasure_check(arm, s, Nc):
    pr = arm.get("probes", {}).get(str(s))
    out = {"N_s": Nc[s], "raw_s": arm["ssdc"][s]}
    if pr:
        out.update(exact=pr["exact"], row=pr["row"], col=pr["col"], chance_exact=pr["chance"]["exact"], chance_row=pr["chance"]["row"])
    if Nc[s] is None or not pr:
        out["pass"] = None; return out
    out["pass"] = bool(Nc[s] <= ERASE_N_MAX and pr["exact"] <= ERASE_EXACT_X * pr["chance"]["exact"]
                       and pr["row"] <= ERASE_AXIS_X * pr["chance"]["row"] and pr["col"] <= ERASE_AXIS_X * pr["chance"]["col"])
    return out

def reading(Nc, s):
    after = [(b, Nc[b]) for b in range(s + 1, N_BLOCKS) if Nc[b] is not None]
    if Nc[s] is None or not after: return {"verdict": "NOT READABLE (intact - floor < %.2f)" % NORM_MIN_RANGE}
    bmax, nmax = max(after, key=lambda t: (t[1], -t[0])); R_ = nmax - Nc[s]
    v = ("REBUILDS" if (R_ >= REBUILD_R and nmax >= REBUILD_NMAX) else ("DOES NOT REBUILD" if R_ < NOREBUILD_R else "PARTIAL"))
    return {"N_s": Nc[s], "N_max": nmax, "N_max_block": bmax, "R": R_, "N_11": Nc[11], "N_final": Nc[FINAL], "verdict": v}

ANALYSIS, TABLES = {}, {"partA": [], "readings": [], "top1": [], "curves": [], "probes": []}
for key in MODELS_RUN:
    R = RESULTS.get(key) or (json.loads(model_file(key).read_text()) if model_file(key).exists() else None)
    if not R or "baseline" not in R.get("ssdc", {}) or floor_name(key) not in R["ssdc"]:
        print(f"{key}: nothing usable saved"); continue
    P = {n: z_[n] for z_ in [np.load(img_file(key))] for n in z_.files} if img_file(key).exists() else {}
    s_ = R["ssdc"]; base = s_["baseline"]["ssdc"]; flo = s_[floor_name(key)]["ssdc"]
    out = {"family": MODEL_SPECS[key]["family"], "label": MODEL_SPECS[key]["label"], "floor_used": floor_name(key),
           "complete": bool(R.get("complete")), "edit_params": R.get("edit_params"), "sites": {}}
    for name, arm in s_.items():
        Nc = norm_curve(arm["ssdc"], base, flo)
        for b in range(N_BLOCKS + 1):
            TABLES["curves"].append({"model": key, "arm": name, "readout": b if b < FINAL else "final", "ssdc": arm["ssdc"][b],
                                     "ssdc_minus_intact": arm["ssdc"][b] - base[b], "N": Nc[b]})
        for b, pr in arm.get("probes", {}).items():
            TABLES["probes"].append({"model": key, "arm": name, "readout": int(b), "exact": pr["exact"], "row": pr["row"], "col": pr["col"],
                                     "exact_x_chance": pr["exact"] / pr["chance"]["exact"], "row_x_chance": pr["row"] / pr["chance"]["row"],
                                     "col_x_chance": pr["col"] / pr["chance"]["col"]})
    for s in sites_of(key):
        o = {"intact_minus_floor_at_s": base[s] - flo[s], "checks": {}, "arms": {}}
        for e in EDITS:
            n = f"{e}_s{s}"
            if n not in s_: continue
            Nc = norm_curve(s_[n]["ssdc"], base, flo)
            o["checks"][e] = erasure_check(s_[n], s, Nc)
            o["arms"][e] = reading(Nc, s)
        if has_rope(key):
            for n, lab in [(f"rotoff_s{s}", "rotoff")] + [(f"{e}_rotoff_s{s}", f"{e}_rotoff") for e in ERASURES]:
                if n in s_: o["arms"][lab] = reading(norm_curve(s_[n]["ssdc"], base, flo), s)
            if f"rotoff_s{s}" in s_:
                Nr = norm_curve(s_[f"rotoff_s{s}"]["ssdc"], base, flo)
                o["rotoff_retention"] = {"N_11": Nr[11], "N_final": Nr[FINAL], "raw_11": s_[f"rotoff_s{s}"]["ssdc"][11],
                                         "intact_raw_11": base[11],
                                         "exact_probe_11": (s_[f"rotoff_s{s}"].get("probes", {}).get("11") or {}).get("exact"),
                                         "intact_exact_probe_11": (s_["baseline"].get("probes", {}).get("11") or {}).get("exact")}
        chosen = next((e for e in ERASURES if (o["checks"].get(e) or {}).get("pass")), None)
        o["chosen_erasure"] = chosen
        readable = any((o["checks"].get(e) or {}).get("pass") is not None for e in ERASURES)
        o["verdict"] = (o["arms"][chosen]["verdict"] if chosen else
                        ("ERASURE INCOMPLETE" if readable else "NOT READABLE (intact - floor < %.2f at s)" % NORM_MIN_RANGE))
        if chosen: o["reading"] = o["arms"][chosen]
        for e in EDITS + (["rotoff"] + [f"{x}_rotoff" for x in ERASURES] if has_rope(key) else []):
            a = o["arms"].get(e)
            if a is None: continue
            ck = o["checks"].get(e, {})
            TABLES["readings"].append({"model": key, "family": out["family"], "site": s, "arm": e, "chosen": e == chosen,
                                       "N_s": a.get("N_s"), "probe_exact_s": ck.get("exact"), "probe_row_s": ck.get("row"),
                                       "probe_col_s": ck.get("col"), "erasure_check": ck.get("pass"), "N_max": a.get("N_max"),
                                       "N_max_block": a.get("N_max_block"), "R": a.get("R"), "N_11": a.get("N_11"),
                                       "N_final": a.get("N_final"), "verdict": a["verdict"]})
        if MODEL_SPECS[key]["classifier"] and "baseline" in P:
            t1 = {}
            for e in EDITS + (["rotoff"] + [f"{x}_rotoff" for x in ERASURES] if has_rope(key) else []):
                n = f"{e}_s{s}" if e in EDITS else (f"rotoff_s{s}" if e == "rotoff" else f"{e.replace('_rotoff', '')}_rotoff_s{s}")
                if n in P:
                    c_, se_ = paired(P["baseline"], P[n]); t1[e] = {"top1": float(P[n].mean() * 100), "cost": c_, "se": se_}
                    TABLES["top1"].append({"model": key, "site": s, "arm": e, "top1": t1[e]["top1"], "intact_top1": float(P["baseline"].mean() * 100),
                                           "cost_points": c_, "se": se_})
            if f"sub_s{s}" in P and f"randsub_s{s}" in P:
                c_, se_ = paired(P[f"randsub_s{s}"], P[f"sub_s{s}"]); t1["sub_minus_randsub"] = {"extra_cost": c_, "se": se_}
            t1["intact"] = float(P["baseline"].mean() * 100)
            t1["floor"] = float(P[floor_name(key)].mean() * 100) if floor_name(key) in P else None
            o["top1"] = t1
        out["sites"][str(s)] = o
    pa = R.get("partA", {})
    for t in ("rep", "acc"):
        for b, v in (pa.get(t) or {}).items():
            TABLES["partA"].append({"model": key, "family": out["family"], "fit": t, "block": int(b), **v,
                                    "cos_rep_vs_ilsvrc": (pa.get("cross_cos") or {}).get(b)})
    ANALYSIS[key] = out

# outcome case (labels only)
rope4 = [k for k in ["naver", "DINOv3", "small", "reg1_gap"] if k in ANALYSIS]
v4 = {k: ANALYSIS[k]["sites"][str(SITE_MAIN)]["verdict"] for k in rope4}
ape_rb = [f"{k}@s{s}" for k in ANALYSIS if MODEL_SPECS[k]["family"] == "ape"
          for s, o in ANALYSIS[k]["sites"].items() if o["verdict"] == "REBUILDS"]
n_not = sum(v in ("DOES NOT REBUILD", "PARTIAL") for v in v4.values())
undet = [k for k, v in v4.items() if v not in ("REBUILDS", "DOES NOT REBUILD", "PARTIAL")]
case = (None if (len(rope4) < 4 or undet) else (1 if n_not == 0 else (2 if n_not <= 2 else 3)))
OUTCOME = {"rule": RULE_CASE, "rope_verdicts_s4": v4, "rope_models_not_rebuilding": n_not, "rope_models_undetermined": undet,
           "ape_verdicts": {f"{k}@s{s}": o["verdict"] for k in ANALYSIS if MODEL_SPECS[k]["family"] == "ape" for s, o in ANALYSIS[k]["sites"].items()},
           "hybrid_verdict_s4": ANALYSIS.get("rope_ape", {}).get("sites", {}).get(str(SITE_MAIN), {}).get("verdict"),
           "ape_models_rebuilding": ape_rb, "case": case,
           "note": ("not all four RoPE models measured" if len(rope4) < 4 else
                    (f"no case: the test could not be read for {undet} (erasure incomplete or range too small); discuss" if undet else
                     ("case 1 holds only if no APE model rebuilds: FLAGGED" if (case == 1 and ape_rb) else
                      ("STOP AND DISCUSS before any text is changed" if case == 3 else ""))))}

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}  ({out['label']}, {out['family']})   floor used for N: {out['floor_used']}\n" + "=" * 78)
    for s, o in out["sites"].items():
        print(f"\n  site s={s}: VERDICT {o['verdict']}  (erasure used: {o['chosen_erasure']}; intact - floor at s = {o['intact_minus_floor_at_s']:.3f})")
        df = pd.DataFrame([{"arm": e, **{k: (r4(v) if isinstance(v, float) else v) for k, v in ck.items() if k not in ("chance_exact", "chance_row")}}
                           for e, ck in o["checks"].items()])
        show(f"  erasure check at the edited state (readout {s}); pass needs N_s <= {ERASE_N_MAX}, exact <= {ERASE_EXACT_X * 100 / 196:.2f}%, "
             f"row/col <= {ERASE_AXIS_X * 100 / 14:.2f}%", df)
        df = pd.DataFrame([{"arm": e, **{k: (r4(v) if isinstance(v, float) else v) for k, v in a.items()}} for e, a in o["arms"].items()])
        show("  reading per arm (R = N_max - N_s over readouts s+1..11)", df)
        if "rotoff_retention" in o: show("  rotations off in blocks s-11, no erasure: how much position the residual stream keeps", o["rotoff_retention"])
        if "top1" in o:
            t1 = o["top1"]
            show(f"  top-1 (points; intact {t1['intact']:.2f}, floor {t1['floor'] if t1['floor'] is None else round(t1['floor'], 2)}; cost = intact - arm, paired SE)",
                 pd.DataFrame([{"arm": e, "top1": round(v["top1"], 2), "cost": round(v["cost"], 2), "se": round(v["se"], 2)}
                               for e, v in t1.items() if isinstance(v, dict) and "top1" in v]))
            if "sub_minus_randsub" in t1:
                print(f"  sub costs {t1['sub_minus_randsub']['extra_cost']:+.2f} +/- {t1['sub_minus_randsub']['se']:.2f} points more than the random subspace of the same size")
show("OUTCOME", OUTCOME)
if TABLES["partA"]:
    pa = pd.DataFrame(TABLES["partA"])
    show("Part A (repack fit): positional variance fraction per block (bias-corrected), split, r99, split-half cosine",
         pa[pa.fit == "rep"].pivot_table(index="block", columns="model", values="pvf_bc").round(4).reset_index())


naver  (naver ViT-B, rope)   floor used for N: floor

  site s=4: VERDICT REBUILDS  (erasure used: sub; intact - floor at s = 0.457)

  erasure check at the edited state (readout 4); pass needs N_s <= 0.1, exact <= 2.04%, row/col <= 10.71%
---------------------------------------------------------------------------------------------------------
     arm     N_s   raw_s  exact    row    col  pass
    mean -0.0335 -0.0024 0.0476 0.1148 0.1118 False
   scale -0.0186  0.0044 0.0449 0.1106 0.1101 False
     sub  0.0519  0.0366 0.0149 0.0909 0.0914  True
 randsub  0.9961  0.4676 0.3262 0.6480 0.6199 False
scramble -0.0186  0.0044 0.4457 0.1798 0.1835 False

  reading per arm (R = N_max - N_s over readouts s+1..11)
---------------------------------------------------------
         arm     N_s  N_max  N_max_block       R    N_11  N_final          verdict
        mean -0.0335 0.9131           11  0.9467  0.9131   0.9545         REBUILDS
       scale -0.0186 0.9191           11  0.9377  0.9191  

## Gate report and export

In [22]:
_all = []
for f in sorted(OUT_DIR.glob("rb6_gates_session_*.json")): _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
complete = {k: bool((RESULTS.get(k) or {}).get("complete")) or (model_file(k).exists() and json.loads(model_file(k).read_text()).get("complete", False)) for k in MODELS_RUN}
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped | complete {complete}")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows(): print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)
export = {"notebook": "RB-6", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "environment": ENV, "repo": str(repo), "repo_commit": REPO_COMMIT,
          "rb0_dir": str(_rb0) if _rb0 else None, "config": {**CFG_FP, "models": MODELS_RUN, "strict": STRICT},
          "rules": {"erasure": RULE_ERASE, "normalisation": RULE_NORM, "reading": RULE_READING, "case": RULE_CASE},
          "analysis": ANALYSIS, "outcome": OUTCOME, "gates": gdf.to_dict("records"),
          "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "complete": complete, "sessions": sorted(set(g["session"] for g in _all))}}
(OUT_DIR / f"rb6_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "rb6_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"rb6_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")

GATE REPORT  (2 session(s), 435 gate calls, latest result per gate shown)

                                                                                                           gate         scope status                                                                                                                                                                                                     detail              session
                                                                                                 CUDA available           env   PASS                                                                                                                                                                                                            2026-11-08T05-56-19Z
                                                                                          torch is 2.10.0+cu128           env   PASS                                                                               

written:
  rb6_DINOv3.json  (61,086 bytes)
  rb6_all_2026-11-08T02-05-10Z.json  (119,597 bytes)
  rb6_all_2026-11-08T05-56-19Z.json  (132,072 bytes)
  rb6_augreg.json  (117,515 bytes)
  rb6_curves.csv  (103,483 bytes)
  rb6_deit3.json  (117,447 bytes)
  rb6_dino.json  (65,393 bytes)
  rb6_environment_2026-11-08T02-05-10Z.json  (1,523 bytes)
  rb6_environment_2026-11-08T05-56-19Z.json  (1,523 bytes)
  rb6_fit_DINOv3.npz  (1,206,314 bytes)
  rb6_fit_augreg.npz  (4,825,190 bytes)
  rb6_fit_deit3.npz  (4,825,190 bytes)
  rb6_fit_dino.npz  (2,412,606 bytes)
  rb6_fit_naver.npz  (2,412,606 bytes)
  rb6_fit_reg1_gap.npz  (2,412,606 bytes)
  rb6_fit_rope_ape.npz  (2,412,606 bytes)
  rb6_fit_sam.npz  (4,825,190 bytes)
  rb6_fit_small.npz  (1,208,382 bytes)
  rb6_fit_supervised.npz  (4,825,190 bytes)
  rb6_gate_report.csv  (32,003 bytes)
  rb6_gates_session_2026-11-08T02-05-10Z.json  (49,951 bytes)
  rb6_gates_session_2026-11-08T05-56-19Z.json  (34,980 bytes)
  rb6_naver.json  (112,987 bytes)
  